<a href="https://colab.research.google.com/github/GaPau/MooCraDee/blob/add-colab-gpu-demo/notebooks/PCS_GPU_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PCS GPU Demo
### A guided Colab pipeline for planetary crater-candidate detection

# Planetary Crater Segmentation (PCS)

PCS is a GPU-enabled crater-candidate detection workflow for planetary images, with a focus on Mercury.

The goal of this notebook is to let you run the PCS pipeline step by step: choose an input image, adjust detection parameters, run the crater-candidate detector, visualize the results, and export a CSV file that can be used as the beginning of a crater-candidate database.

PCS builds on the MooCraDee project and uses the Segment Anything Model (SAM) together with computer vision and geometric filtering. SAM helps generate segmentation masks, and PCS filters those masks to keep structures that look crater-like.

This notebook is not meant to replace expert planetary analysis. Instead, it is a reproducible tool for exploring crater-candidate detection, testing parameters, and preparing outputs for further inspection.


# 1. How This Notebook Works

This notebook is designed as a guided pipeline. Each section explains what is happening, runs one part of the workflow, and prepares the next step.

By the end of the notebook, you will be able to:

* check that GPU acceleration is available,
* install the required dependencies,
* clone the PCS/MooCraDee repository,
* download the SAM model checkpoint,
* verify that the required files are available,
* run PCS on the default Mercury image or on your own uploaded planetary image,
* adjust detection parameters such as radius range, circularity, area, and SAM mask quality,
* visualize detected crater candidates,
* load the output CSV as a crater-candidate database,
* export results for future analysis.

**Pro Tip: Use GPU Acceleration**

PCS uses SAM, and SAM runs much faster with a GPU. In Google Colab, go to:

`Runtime` → `Change runtime type` → `Hardware accelerator` → `T4 GPU`

Then click `Save`.

**Where are the files saved?**

Google Colab runs in a temporary cloud environment. Files created during this session are saved under `/content/`. They are not automatically saved to Google Drive. If the runtime disconnects or restarts, the outputs may be deleted. At the end of this notebook, you will have the option to download the results or copy them to Google Drive.

Let's begin.


# 2. Runtime and GPU Setup

PCS uses the Segment Anything Model (SAM), which runs much faster with GPU acceleration. This section checks whether the current Colab runtime has access to a GPU.

If no GPU is detected, go to:

`Runtime` → `Change runtime type` → `Hardware accelerator` → `T4 GPU`

Then click `Save` and run this section again.


In [8]:
import torch

if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("GPU not available.")
    print("Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")

GPU available: Tesla T4


# 3. Install Dependencies

This section installs the Python libraries needed to run PCS.

PCS uses:

* **PyTorch** to run SAM with GPU support.
* **Segment Anything** to generate segmentation masks.
* **OpenCV** for image processing, contours, circularity filtering, and drawing detections.
* **Pillow** for image loading and splitting.
* **NumPy** for numerical operations.
* **Pandas** for loading and inspecting CSV outputs.
* **Matplotlib** for plots and result analysis.

The installation may take a few minutes the first time you run it.


In [2]:
!pip install opencv-python pillow matplotlib numpy torch torchvision
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-m831p46e
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-m831p46e
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36592 sha256=cdb1a8e83b703fd58f2d13b139f253fda99b9ad7f9f0f0d22bbbe6d467a78ccd
  Stored in directory: /tmp/pip-ephem-wheel-cache-_tyrq1pn/wheels/29/82/ff/04e2be9805a1cb48bec0b85b5a6da6b63f647645750a0e42d4
Successfully built segment_anything


# 4. Clone Repository and Set Working Directory

This section downloads the PCS/MooCraDee project from GitHub into the Colab runtime.

The repository is saved inside Colab’s temporary file system, usually under `/content/MooCraDee`. This means the files are available while the current Colab session is active, but they are not automatically saved to Google Drive.

During development, this notebook may use a testing branch. After the Colab workflow is merged into the main project, the notebook should run from the `main` branch.



In [11]:
from pathlib import Path

repo_url = "https://github.com/GaPau/MooCraDee.git"
repo_dir = Path("/content/MooCraDee")

branch_name = "add-colab-gpu-demo"  # CHANGEE to "main" after merging

if not repo_dir.exists():
    print("Cloning repository...")
    !git clone {repo_url} {repo_dir}
else:
    print("Repository already exists.")

%cd /content/MooCraDee

!git checkout {branch_name}

Repository already exists.
/content/MooCraDee
M	mercury_split_6/part_1/input.png
M	mercury_split_6/part_2/input.png
M	mercury_split_6/part_3/input.png
M	mercury_split_6/part_4/input.png
M	mercury_split_6/part_5/input.png
M	mercury_split_6/part_6/input.png
Already on 'add-colab-gpu-demo'
Your branch is up to date with 'origin/add-colab-gpu-demo'.


In [4]:
!git checkout add-colab-gpu-demo

Branch 'add-colab-gpu-demo' set up to track remote branch 'add-colab-gpu-demo' from 'origin'.
Switched to a new branch 'add-colab-gpu-demo'


# 5. Download SAM Checkpoint

PCS uses the Segment Anything Model (SAM) from Meta AI. To run SAM, the notebook needs a model checkpoint file.

The checkpoint is not stored directly in the repository because model files can be large. Instead, this section downloads the SAM ViT-B checkpoint into the Colab runtime when needed.

If the checkpoint already exists, the notebook will skip the download.


In [12]:
from pathlib import Path

checkpoint_path = Path("sam_vit_b_01ec64.pth")

if not checkpoint_path.exists():
    print("Downloading SAM ViT-B checkpoint...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    print("Download complete.")
else:
    print("SAM checkpoint already exists.")

SAM checkpoint already exists.


# 6. Verify Required Files

Before running PCS, this section checks that the required project files are available in the Colab runtime.

PCS needs:

* `mercury.jpg` as the default Mercury test image,
* `run_pipeline.py` to run the full split-image workflow,
* `deep_moocrade.py` to run crater-candidate detection on one image,
* `split_image.py` to divide large images into smaller sections,
* `sam_vit_b_01ec64.pth` as the SAM model checkpoint.

If one of these files is missing, the notebook will show it here before the pipeline runs.


In [13]:
from pathlib import Path

required_files = [
    "mercury.jpg",
    "run_pipeline.py",
    "deep_moocrade.py",
    "split_image.py",
    "sam_vit_b_01ec64.pth"
]

for file in required_files:
    if Path(file).exists():
        print(f"Found: {file}")
    else:
        print(f"Missing: {file}")

Found: mercury.jpg
Found: run_pipeline.py
Found: deep_moocrade.py
Found: split_image.py
Found: sam_vit_b_01ec64.pth


# 7. Select Input Image

PCS can run in two ways:

1. Use the default Mercury image included in the repository.
2. Upload your own planetary image from your computer.

For large planetary images, the split-image pipeline is recommended because it divides the image into smaller parts before detection. For smaller test images, the single-image mode can run directly without splitting.

Supported image formats include `.jpg`, and `.png`.



In [16]:
from google.colab import files
from pathlib import Path

use_uploaded_image = False  # Change to True if you want to upload your own image

if use_uploaded_image:
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]
    print("Uploaded image:", image_path)
else:
    image_path = "mercury.jpg"
    print("Using default image:", image_path)

image_path = Path(image_path)

Using default image: mercury.jpg


# 8. Configure Detection Parameters

This section controls how PCS filters crater candidates.

SAM generates segmentation masks, but not every mask is a crater. PCS uses geometric and filters to keep masks that look more crater-like.

You can adjust these values to test how the results change:

* `min_radius`: smallest crater candidate radius allowed, in pixels.
* `max_radius`: largest crater candidate radius allowed, in pixels.
* `min_circularity`: how circular a candidate must be. Higher values are stricter.
* `min_area`: removes masks that are too small.
* `points_per_side`: controls how densely SAM samples the image. Higher values may detect more masks, but can be slower.
* `pred_iou`: SAM’s predicted mask quality threshold.
* `stability`: how stable the SAM mask must be.
* `iou_dedup`: removes duplicate crater candidates that overlap too much.

Start with the default values below. Then try changing one parameter at a time to see how the detection results change.


In [17]:
# Crater geometry filters
min_radius = 20
max_radius = 260
min_circularity = 0.35
min_area = 600

# SAM mask generation parameters
points_per_side = 64
pred_iou = 0.80
stability = 0.85

# Duplicate removal
iou_dedup = 0.12

print("Detection parameters configured.")

Detection parameters configured.


# 9. Choose Pipeline Mode

PCS can run in two modes.

**Mode A: Full Mercury pipeline with image splitting**

Use this mode for the default Mercury image or for large planetary images. The image is divided into smaller parts, PCS runs detection on each part, and the results are combined into one CSV file.

**Mode B: Single-image detection without splitting**

Use this mode for smaller test images or uploaded planetary images. PCS runs directly on the selected image without dividing it into parts.

For a first test, use `single_image` mode. For the full Mercury workflow, use `split_image` mode.


In [19]:
pipeline_mode = "split_image"  # Options: "single_image" or "split_image"

# Number of image parts for split_image mode.
# This must be a positive even number: 2, 4, 6, 8, ...
n_splits = 4

print("Selected pipeline mode:", pipeline_mode)

if pipeline_mode == "split_image":
    print("The image will be divided into", n_splits, "parts before detection.")
elif pipeline_mode == "single_image":
    print("PCS will run directly on the selected image without splitting.")
else:
    print("Invalid mode. Use 'single_image' or 'split_image'.")

Selected pipeline mode: split_image
The image will be divided into 4 parts before detection.


# 10. Run PCS Pipeline

This section runs PCS using the selected image, detection parameters, and pipeline mode.

If `pipeline_mode = "single_image"`, PCS runs directly on the selected image and creates:

* an output image with detected crater candidates,
* a CSV file with candidate coordinates, radius, and score.

If `pipeline_mode = "split_image"`, PCS divides the image into multiple parts, runs detection on each part, and combines the detections into one CSV file.

This step may take several minutes depending on the image size, GPU availability, and SAM parameters.


In [20]:
from pathlib import Path

image_name = image_path.stem

if pipeline_mode == "single_image":
    output_image = Path(f"{image_name}_detected.png")
    output_csv = Path(f"{image_name}_craters.csv")

    command = f"""
    python deep_moocrade.py {image_path} \
      --ckpt sam_vit_b_01ec64.pth \
      --out {output_image} \
      --csv {output_csv} \
      --min_radius {min_radius} \
      --max_radius {max_radius} \
      --min_circularity {min_circularity} \
      --min_area {min_area} \
      --pps {points_per_side} \
      --pred_iou {pred_iou} \
      --stability {stability} \
      --iou_dedup {iou_dedup}
    """

    print("Running PCS in single-image mode...")
    print("Input image:", image_path)
    !{command}

elif pipeline_mode == "split_image":
    output_dir = Path(f"{image_name}_split_{n_splits}")
    output_csv = output_dir / "all_craters.csv"

    print("Running PCS in split-image mode...")
    print("Input image:", image_path)
    print("Number of splits:", n_splits)

    # Current run_pipeline.py uses the default mercury.jpg workflow.
    # For now, split-image mode is recommended for the default Mercury image.
    !python run_pipeline.py {n_splits}

else:
    raise ValueError("Invalid pipeline_mode. Use 'single_image' or 'split_image'.")

Running PCS in split-image mode...
Input image: mercury.jpg
Number of splits: 4
Guardado: mercury_split_4/part_1/input.png
Guardado: mercury_split_4/part_2/input.png
Guardado: mercury_split_4/part_3/input.png
Guardado: mercury_split_4/part_4/input.png

Done. Created 4 folders in: mercury_split_4

Running detector on part 1
Input: mercury_split_4/part_1/input.png
Output image: mercury_split_4/part_1/detected.png
CSV: mercury_split_4/part_1/radii.csv
Using GPU/CUDA: Tesla T4
Device: cuda
Circulos finales: 64
1: centro=(876.7,932.8)  radio=23.8px  score=1.891
2: centro=(1812.5,523.0)  radio=20.2px  score=1.878
3: centro=(1011.5,910.2)  radio=29.9px  score=1.866
4: centro=(940.5,427.5)  radio=39.7px  score=1.864
5: centro=(838.0,372.0)  radio=21.1px  score=1.857
6: centro=(541.9,764.1)  radio=21.5px  score=1.855
7: centro=(1223.6,865.3)  radio=23.8px  score=1.855
8: centro=(1362.4,507.9)  radio=28.2px  score=1.854
9: centro=(1398.5,466.0)  radio=27.4px  score=1.854
10: centro=(1144.5,953.5

In [7]:
!python run_pipeline.py 6

Guardado: mercury_split_6/part_1/input.png
Guardado: mercury_split_6/part_2/input.png
Guardado: mercury_split_6/part_3/input.png
Guardado: mercury_split_6/part_4/input.png
Guardado: mercury_split_6/part_5/input.png
Guardado: mercury_split_6/part_6/input.png

Done. Created 6 folders in: mercury_split_6

Running detector on part 1
Input: mercury_split_6/part_1/input.png
Output image: mercury_split_6/part_1/detected.png
CSV: mercury_split_6/part_1/radii.csv
Traceback (most recent call last):
  File "/usr/lib/python3.12/subprocess.py", line 550, in run
    stdout, stderr = process.communicate(input, timeout=timeout)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/subprocess.py", line 1196, in communicate
    stdout = self.stdout.read()
             ^^^^^^^^^^^^^^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/MooCraDee/run_pipeline.py", line 173, in 

Donde estas y que se creo, despues de la pipeline


In [ ]:
!pwd
!ls
!ls mercury_split_6

/content/MooCraDee/MooCraDee
art2moon.jpg	  mercury.jpg	   __pycache__		 split_image.py
assets		  mercury_split_2  README.md
deep_moocrade.py  mercury_split_6  run_pipeline.py
examples	  notebooks	   sam_vit_b_01ec64.pth
all_craters.csv  part_1  part_2  part_3  part_4  part_5  part_6


## Run PCS on a Single Test Image

In [ ]:
!python deep_moocrade.py art2moon.jpg \
  --ckpt sam_vit_b_01ec64.pth \
  --out art2moon_detected.png \
  --csv art2moon_craters.csv \
  --min_radius 20 \
  --max_radius 260 \
  --min_circularity 0.35 \
  --min_area 600 \
  --pps 64 \
  --pred_iou 0.80 \
  --stability 0.85 \
  --iou_dedup 0.12

Using GPU/CUDA: Tesla T4
Device: cuda
Circulos finales: 174
1: centro=(1647.2,185.4)  radio=31.0px  score=1.904
2: centro=(1597.5,82.0)  radio=22.6px  score=1.888
3: centro=(1244.0,42.0)  radio=28.2px  score=1.882
4: centro=(1343.8,217.6)  radio=24.0px  score=1.877
5: centro=(1866.5,171.5)  radio=29.8px  score=1.869
6: centro=(1621.0,479.5)  radio=30.3px  score=1.867
7: centro=(1216.0,175.5)  radio=80.7px  score=1.862
8: centro=(1016.0,307.0)  radio=38.8px  score=1.856
9: centro=(1591.5,1167.0)  radio=25.5px  score=1.855
10: centro=(1648.9,356.4)  radio=23.2px  score=1.855
11: centro=(1245.6,417.8)  radio=22.9px  score=1.855
12: centro=(1760.7,662.7)  radio=23.8px  score=1.842
13: centro=(1848.5,323.5)  radio=23.5px  score=1.841
14: centro=(1331.1,324.1)  radio=30.4px  score=1.840
15: centro=(1433.5,494.7)  radio=30.0px  score=1.838
16: centro=(1329.0,410.0)  radio=22.0px  score=1.836
17: centro=(697.6,378.4)  radio=22.8px  score=1.833
18: centro=(1537.0,543.5)  radio=24.4px  score=1.8

## 5. Set Detection Parameters

## 6. Run PCS Pipeline


## 7. Visualize Results

## 8. Export Crater Candidate Database

# New Section

In [ ]:
from google.colab import drive
drive.mount('/content/drive')